In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 3.1 Introduction to Tree-Based Models
- Three models, one shared API: `instantiate -> fit -> predict`
- Decision Trees, Random Forests, XGBoost
- The practical toolkit for tabular data

## Why These Three Models?
| Model | Strategy | Key Strength |
|:------|:---------|:-------------|
| **Decision Tree** | Single tree | Highly interpretable |
| **Random Forest** | Many trees in parallel (bagging) | Robust, reduces overfitting |
| **XGBoost** | Trees in sequence (boosting) | Top performance |

## The Common Pattern
```python
# Same 3 lines for ALL tree-based models:
model = ModelClass(hyperparameters)    # 1. Instantiate
model.fit(X_train, y_train)           # 2. Fit
predictions = model.predict(X_test)   # 3. Predict
```
**Only Step 1 changes** between models!

## Let's See the Pattern in Action

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                              use_label_encoder=False, eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    print(f"{name}:")
    print(f"  model.fit(X_train, y_train)")
    print(f"  model.predict(X_test)")
    print(f"  model.predict_proba(X_test)")
    print()

print("Same three lines. Different model, same interface.")

## What is a Decision Tree?
- Learns yes/no rules from data (like a flowchart)
- Each node asks a question about a feature
- Each leaf provides a prediction
- **Higher Ed example**: "Is GPA < 2.0?" -> "Did they fail courses?" -> Risk level

## Anatomy of a Decision Tree

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

nodes = {
    'root': (0.5, 1.0), 'left1': (0.25, 0.7), 'right1': (0.75, 0.7),
    'left2': (0.125, 0.4), 'right2': (0.375, 0.4),
    'left3': (0.625, 0.4), 'right3': (0.875, 0.4)
}

edges = [('root','left1'),('root','right1'),('left1','left2'),('left1','right2'),
         ('right1','left3'),('right1','right3')]
for s, e in edges:
    fig.add_trace(go.Scatter(x=[nodes[s][0],nodes[e][0]], y=[nodes[s][1],nodes[e][1]],
                             mode='lines', line=dict(color='gray',width=2), showlegend=False))

fig.add_trace(go.Scatter(
    x=[nodes[n][0] for n in ['root','left1','right1']],
    y=[nodes[n][1] for n in ['root','left1','right1']],
    mode='markers+text', marker=dict(size=50, color='lightblue', line=dict(color='blue',width=2)),
    text=['GPA_1 <= 3.5?', 'DFW > 0.3?', 'UNITS < 12?'],
    textposition='middle center', textfont=dict(size=10), name='Decision Nodes'))

fig.add_trace(go.Scatter(
    x=[nodes[n][0] for n in ['left2','right2','left3','right3']],
    y=[nodes[n][1] for n in ['left2','right2','left3','right3']],
    mode='markers+text', marker=dict(size=50, color='lightgreen', symbol='square',
    line=dict(color='green',width=2)),
    text=['Not Enrolled','Enrolled','Not Enrolled','Enrolled'],
    textposition='middle center', textfont=dict(size=9), name='Leaf Nodes'))

fig.add_annotation(x=0.35, y=0.88, text='Yes', showarrow=False, font=dict(color='green'))
fig.add_annotation(x=0.65, y=0.88, text='No', showarrow=False, font=dict(color='red'))

fig.update_layout(title='Anatomy of a Decision Tree', height=450,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

## How Trees Make Splits: Gini Impurity
- **Gini = 0** -> pure node (all one class)
- **Gini = 0.5** -> maximum impurity (50/50)
- Tree picks the split that reduces Gini the most
- Greedy algorithm — locally optimal at each step

## Controlling Tree Overfitting
| Parameter | What It Does | Typical Range |
|:----------|:-------------|:-------------|
| `max_depth` | Maximum tree depth | 3-15 |
| `min_samples_split` | Min samples to split | 5-50 |
| `min_samples_leaf` | Min samples in leaf | 3-20 |
| `max_features` | Features per split | `'sqrt'`, `'log2'` |

## Random Forest: Many Trees > One Tree
- Single trees are unstable — small data changes = very different tree
- Random Forest: train many trees, average predictions

### The Bagging Strategy
```
Training Data
  +---> Bootstrap Sample 1 ---> Tree 1 ---+
  +---> Bootstrap Sample 2 ---> Tree 2 ---+--> Majority Vote
  +---> Bootstrap Sample N ---> Tree N ---+
```

## Random Forest Key Parameters
| Parameter | What It Does | Typical Range |
|:----------|:-------------|:-------------|
| `n_estimators` | Number of trees | 100-500 |
| `max_depth` | Max depth per tree | 8-20 or None |
| `max_features` | Features per split | `'sqrt'` (default) |
| `class_weight` | Handle imbalance | `'balanced'` |

## XGBoost: Sequential Error Correction
- Trees trained **sequentially** — each corrects previous errors
- Start with simple prediction -> calculate errors -> train tree on errors -> repeat

```
Data --> Tree 1 --> Errors --> Tree 2 --> Errors --> Tree 3 --> ...
```

**Key:** F_m(x) = F_{m-1}(x) + learning_rate * new_tree(x)

## Why XGBoost Dominates Tabular Data
- Built-in regularization (L1 + L2)
- Handles missing values automatically
- Column subsampling for diversity
- Early stopping to prevent overfitting

## XGBoost Key Parameters
| Parameter | What It Does | Typical Range |
|:----------|:-------------|:-------------|
| `n_estimators` | Boosting rounds | 100-1000 |
| `learning_rate` | Step size | 0.01-0.3 |
| `max_depth` | Depth per tree (shallow!) | 3-8 |
| `subsample` | Row sampling | 0.7-1.0 |
| `colsample_bytree` | Column sampling | 0.7-1.0 |

## Comparing the Three
| Aspect | Decision Tree | Random Forest | XGBoost |
|:-------|:-------------|:-------------|:--------|
| Strategy | Single tree | Parallel trees | Sequential trees |
| Reduces | — | Variance | Bias |
| Interpretability | Excellent | Moderate | Lower |
| Preprocessing | None needed | None needed | None needed |
| Typical use | Stakeholder communication | Reliable default | Best performance |

## Key Takeaways
- All three follow `instantiate -> fit -> predict`
- **No preprocessing needed** (no scaling!)
- Decision Trees: interpretable but overfit
- Random Forests: robust via bagging
- XGBoost: best performance via boosting
- **Next:** 3.2 Building Tree-Based Models in Practice